In [141]:
import torch
import torch.nn as nn
from torchvision.datasets import ImageFolder
from torchvision.transforms import v2
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader


In [142]:
train_path = r'./dataset/train'
test_path = r'./dataset/valid'

In [143]:

transforms = v2.Compose([
    v2.ToImage(),
    v2.CenterCrop(size=(256, 256)),
    ##v2.RandomResizedCrop(size=(128, 128), antialias=True),
    v2.RandomHorizontalFlip(p=0.5),
    v2.ToDtype(torch.float32,scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

In [144]:
train_dataset =ImageFolder(train_path,transforms)

In [145]:
data_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

残差神经网络

In [146]:
class ResBlock(nn.Module):
    def __init__(self,in_channels,hid_channels):
        super().__init__()
        self.inchannels = in_channels
        self.cnn = nn.Sequential(
            nn.Conv2d(in_channels,hid_channels,kernel_size=3,padding=1),
            nn.BatchNorm2d(hid_channels),
            nn.SiLU(),
            nn.Conv2d(hid_channels,hid_channels,kernel_size=5,padding=2),
            nn.BatchNorm2d(hid_channels),
            nn.SiLU(),
            nn.Conv2d(hid_channels,in_channels,kernel_size=3,padding=1)
        )
    def forward(self,x):
        return x + self.cnn(x)
    
class ResNet(nn.Module):
    def __init__(self,in_channels,out_channels,res_channels,hid_channels,num_res):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.res_channels = res_channels
        self.hid_channels = hid_channels
        self.num_res = num_res
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels,res_channels,kernel_size=3,padding=1),
            nn.SiLU()
        )
        self.decoder=nn.Conv2d(res_channels,out_channels,kernel_size=3,padding=1)
        self.hid_res_layers = nn.Sequential(*[ResBlock(res_channels,hid_channels) for _ in range(num_res)])
    
    def forward(self,x):
        y = self.encoder(x)
        y = self.hid_res_layers(y)
        y = self.decoder(y)
        print(y.shape)
        return y.mean(dim=(-2,-1))


In [147]:
x= torch.randn([1,3,256,256])
resnet = ResNet(3,20,16,32,20)
y = resnet(x)
print(y.shape)

torch.Size([1, 20, 256, 256])
torch.Size([1, 20])


In [ ]:
def train(model,dataloader,epoch,lr):
    optim = torch.optim.Adam(model.parameters(),lr=lr)
    loss_fun = nn.CrossEntropyLoss()
    loss_values = torch.zeros(epoch)
    for i in range(epoch):
        model.train()
        for data,label in dataloader:
            optim.zero_grad()
            pred = model(data)
            loss = loss_fun(pred,label)
            loss.backward()
            print(i,loss.item())
            


: 

In [ ]:
train(resnet,data_loader,10,1e-3)